# Telco Customer Churn - Step 3: Feature Engineering, Baseline & Balanced Modeling

This notebook implements the complete feature preprocessing pipeline, trains 3 baseline models, and addresses the class imbalance problem using cost-sensitive / balanced training.

### Pipeline Structure:
1. **Load Cleaned Data**: `data/cleaned_churn.csv`
2. **Feature / Target Separation**: $X$ (predictors), $y$ (`Churn`)
3. **One-Hot Encoding**: `drop_first=True` across 15 categoricals
4. **Stratified 80/20 Train/Test Split**: Preserving ~26.5% churn distribution (`random_state=42`)
5. **Feature Scaling**: `StandardScaler` fitted **only** on `X_train` to prevent data leakage
6. **Baseline Models (Unweighted)**: Logistic Regression, Random Forest, Gradient Boosting
7. **Balanced Models**: Re-training all three models with `class_weight='balanced'` and `sample_weight`
8. **Side-by-Side Comparison**: In-depth benchmark (Accuracy, Precision, Recall, F1, ROC-AUC, and Confusion Matrices)

In [11]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 1. Load Cleaned Dataset

In [12]:
csv_path = Path("../data/cleaned_churn.csv")
if not csv_path.exists():
    csv_path = Path("data/cleaned_churn.csv")

df = pd.read_csv(csv_path)
print(f"Loaded dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()

Loaded dataset: 7,043 rows, 20 columns


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


## 2. Separate Features (X) and Target (y)

In [13]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape:   {y.shape}")
print("\nClass balance:")
print(pd.Series(y).value_counts(normalize=True).map(lambda p: f"{p*100:.2f}%"))

Feature matrix X shape: (7043, 19)
Target vector y shape:   (7043,)

Class balance:
Churn
0    73.46%
1    26.54%
Name: proportion, dtype: str


## 3. One-Hot Encode Categoricals (`drop_first=True`)

We encode all multi-level and binary categorical variables, dropping the reference level (`drop_first=True`) to prevent collinearity.

In [14]:
categorical_cols = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod"
]

X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True, dtype=int)
print(f"Encoded feature count: {X_encoded.shape[1]} columns")
print("Columns:", list(X_encoded.columns))

Encoded feature count: 23 columns
Columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_Yes', 'OnlineBackup_Yes', 'DeviceProtection_Yes', 'TechSupport_Yes', 'StreamingTV_Yes', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


## 4. Stratified 80/20 Train/Test Split

Preserving identical ~26.5% churn distribution across both sets using `stratify=y`.

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set : {X_train.shape[0]:,} samples (Churn rate: {y_train.mean()*100:.2f}%)")
print(f"Testing set  : {X_test.shape[0]:,} samples (Churn rate: {y_test.mean()*100:.2f}%)")

Training set : 5,634 samples (Churn rate: 26.54%)
Testing set  : 1,409 samples (Churn rate: 26.54%)


## 5. Scale Numeric Features (`StandardScaler`)

> **Data Leakage Guard**: Fit `StandardScaler` exclusively on `X_train`, then transform both train and test.

In [16]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

print("Training numeric stats after scaling:")
display(X_train_scaled[numeric_cols].describe().round(4))

Training numeric stats after scaling:


,tenure,MonthlyCharges,TotalCharges
count,5634.0000,5634.0000,5634.0000
mean,-0.0000,-0.0000,0.0000
std,1.0001,1.0001,1.0001
min,-1.3223,-1.5440,-1.0089
25%,-0.9560,-0.9712,-0.8321
50%,-0.1419,0.1848,-0.3968
75%,0.9165,0.8319,0.6742
max,1.6085,1.7859,2.8019


## 6. Baseline Models (Default / Unweighted)

We train three foundational algorithms:
1. **Logistic Regression**
2. **Random Forest Classifier**
3. **Gradient Boosting Classifier**

In [17]:
def evaluate(model, X, y):
    preds = model.predict(X)
    probs = model.predict_proba(X)[:, 1]
    return {
        "Accuracy": accuracy_score(y, preds),
        "Precision": precision_score(y, preds),
        "Recall": recall_score(y, preds),
        "F1-Score": f1_score(y, preds),
        "ROC-AUC": roc_auc_score(y, probs),
    }

# Model 1: Logistic Regression
lr_base = LogisticRegression(max_iter=1000, random_state=42)
lr_base.fit(X_train_scaled, y_train)

# Model 2: Random Forest
rf_base = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_base.fit(X_train_scaled, y_train)

# Model 3: Gradient Boosting
gb_base = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
gb_base.fit(X_train_scaled, y_train)

baseline_df = pd.DataFrame({
    "Logistic Regression": evaluate(lr_base, X_test_scaled, y_test),
    "Random Forest": evaluate(rf_base, X_test_scaled, y_test),
    "Gradient Boosting": evaluate(gb_base, X_test_scaled, y_test),
})

print("=== BASELINE PERFORMANCE (Unweighted) ===")
display(baseline_df.round(4))

=== BASELINE PERFORMANCE (Unweighted) ===


,Logistic Regression,Random Forest,Gradient Boosting
Accuracy,0.8055,0.8070,0.8027
Precision,0.6582,0.6783,0.6622
Recall,0.5561,0.5187,0.5241
F1-Score,0.6029,0.5879,0.5851
ROC-AUC,0.8422,0.8428,0.8433


## 7. Class-Balanced Models (`class_weight='balanced'` / `sample_weight`)

Because the baseline models only achieved ~52–56% recall on churners (missing almost half of actual churners), we now re-train all three models using class-weight balancing:
- **Logistic Regression**: `class_weight='balanced'`
- **Random Forest**: `class_weight='balanced'`
- **Gradient Boosting**: `sample_weight=compute_sample_weight('balanced', y_train)`

In [18]:
# Balanced Model 1: Logistic Regression
lr_bal = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr_bal.fit(X_train_scaled, y_train)

# Balanced Model 2: Random Forest
rf_bal = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight="balanced", random_state=42)
rf_bal.fit(X_train_scaled, y_train)

# Balanced Model 3: Gradient Boosting
sample_weights_train = compute_sample_weight("balanced", y_train)
gb_bal = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
gb_bal.fit(X_train_scaled, y_train, sample_weight=sample_weights_train)

balanced_df = pd.DataFrame({
    "Logistic Regression (Bal)": evaluate(lr_bal, X_test_scaled, y_test),
    "Random Forest (Bal)": evaluate(rf_bal, X_test_scaled, y_test),
    "Gradient Boosting (Bal)": evaluate(gb_bal, X_test_scaled, y_test),
})

print("=== BALANCED MODEL PERFORMANCE ===")
display(balanced_df.round(4))

=== BALANCED MODEL PERFORMANCE ===


,Logistic Regression (Bal),Random Forest (Bal),Gradient Boosting (Bal)
Accuracy,0.7402,0.7587,0.7452
Precision,0.5069,0.5316,0.5132
Recall,0.7861,0.7647,0.7781
F1-Score,0.6164,0.6272,0.6185
ROC-AUC,0.8419,0.8417,0.8388


## 8. Side-by-Side Comparison: Baseline vs Balanced

Compare all 6 configurations directly to inspect the trade-offs between precision, recall, and overall F1-score.

In [19]:
comparison_table = pd.DataFrame({
    "Model": [
        "Logistic Regression (Baseline)", "Logistic Regression (Balanced)",
        "Random Forest (Baseline)", "Random Forest (Balanced)",
        "Gradient Boosting (Baseline)", "Gradient Boosting (Balanced)",
    ],
    "Accuracy": [
        baseline_df.loc["Accuracy", "Logistic Regression"],
        balanced_df.loc["Accuracy", "Logistic Regression (Bal)"],
        baseline_df.loc["Accuracy", "Random Forest"],
        balanced_df.loc["Accuracy", "Random Forest (Bal)"],
        baseline_df.loc["Accuracy", "Gradient Boosting"],
        balanced_df.loc["Accuracy", "Gradient Boosting (Bal)"],
    ],
    "Precision": [
        baseline_df.loc["Precision", "Logistic Regression"],
        balanced_df.loc["Precision", "Logistic Regression (Bal)"],
        baseline_df.loc["Precision", "Random Forest"],
        balanced_df.loc["Precision", "Random Forest (Bal)"],
        baseline_df.loc["Precision", "Gradient Boosting"],
        balanced_df.loc["Precision", "Gradient Boosting (Bal)"],
    ],
    "Recall": [
        baseline_df.loc["Recall", "Logistic Regression"],
        balanced_df.loc["Recall", "Logistic Regression (Bal)"],
        baseline_df.loc["Recall", "Random Forest"],
        balanced_df.loc["Recall", "Random Forest (Bal)"],
        baseline_df.loc["Recall", "Gradient Boosting"],
        balanced_df.loc["Recall", "Gradient Boosting (Bal)"],
    ],
    "F1-Score": [
        baseline_df.loc["F1-Score", "Logistic Regression"],
        balanced_df.loc["F1-Score", "Logistic Regression (Bal)"],
        baseline_df.loc["F1-Score", "Random Forest"],
        balanced_df.loc["F1-Score", "Random Forest (Bal)"],
        baseline_df.loc["F1-Score", "Gradient Boosting"],
        balanced_df.loc["F1-Score", "Gradient Boosting (Bal)"],
    ],
    "ROC-AUC": [
        baseline_df.loc["ROC-AUC", "Logistic Regression"],
        balanced_df.loc["ROC-AUC", "Logistic Regression (Bal)"],
        baseline_df.loc["ROC-AUC", "Random Forest"],
        balanced_df.loc["ROC-AUC", "Random Forest (Bal)"],
        baseline_df.loc["ROC-AUC", "Gradient Boosting"],
        balanced_df.loc["ROC-AUC", "Gradient Boosting (Bal)"],
    ],
})

display(comparison_table.round(4))

print("\n--- Churn Detection Impact (Total Churners in Test Set: 374) ---")
all_models = {
    "Logistic Regression (Base)": lr_base,
    "Logistic Regression (Bal)":  lr_bal,
    "Random Forest (Base)":       rf_base,
    "Random Forest (Bal)":        rf_bal,
    "Gradient Boosting (Base)":   gb_base,
    "Gradient Boosting (Bal)":    gb_bal,
}

for name, model in all_models.items():
    cm = confusion_matrix(y_test, model.predict(X_test_scaled))
    caught = cm[1, 1]
    missed = cm[1, 0]
    print(f"{name:<30} -> Caught: {caught:3d} ({caught/374*100:.1f}%) | Missed: {missed:3d} ({missed/374*100:.1f}%)")

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression (Baseline),0.8055,0.6582,0.5561,0.6029,0.8422
1,Logistic Regression (Balanced),0.7402,0.5069,0.7861,0.6164,0.8419
2,Random Forest (Baseline),0.8070,0.6783,0.5187,0.5879,0.8428
3,Random Forest (Balanced),0.7587,0.5316,0.7647,0.6272,0.8417
4,Gradient Boosting (Baseline),0.8027,0.6622,0.5241,0.5851,0.8433
5,Gradient Boosting (Balanced),0.7452,0.5132,0.7781,0.6185,0.8388



--- Churn Detection Impact (Total Churners in Test Set: 374) ---
Logistic Regression (Base)     -> Caught: 208 (55.6%) | Missed: 166 (44.4%)
Logistic Regression (Bal)      -> Caught: 294 (78.6%) | Missed:  80 (21.4%)
Random Forest (Base)           -> Caught: 194 (51.9%) | Missed: 180 (48.1%)
Random Forest (Bal)            -> Caught: 286 (76.5%) | Missed:  88 (23.5%)
Gradient Boosting (Base)       -> Caught: 196 (52.4%) | Missed: 178 (47.6%)
Gradient Boosting (Bal)        -> Caught: 291 (77.8%) | Missed:  83 (22.2%)


## 9. Top Feature Importance Comparison

Inspect the most influential features identified across linear and tree-based ensembles.

In [20]:
feat_df = pd.DataFrame({
    "Feature": X_encoded.columns,
    "LogReg Coef": lr_bal.coef_[0],
    "Random Forest Imp": rf_bal.feature_importances_,
    "GradBoost Imp": gb_bal.feature_importances_,
})

print("Top 8 Features by Gradient Boosting Importance:")
display(feat_df.sort_values(by="GradBoost Imp", ascending=False).head(8).round(4))

Top 8 Features by Gradient Boosting Importance:


,Feature,LogReg Coef,Random Forest Imp,GradBoost Imp
18,Contract_Two year,-1.4164,0.0970,0.2578
17,Contract_One year,-0.7190,0.0447,0.1544
1,tenure,-1.1568,0.1740,0.1317
3,TotalCharges,0.4858,0.1455,0.1023
9,InternetService_Fiber optic,1.0785,0.0882,0.0832
2,MonthlyCharges,-0.3677,0.1274,0.0759
21,PaymentMethod_Electronic check,0.4058,0.0561,0.0638
10,InternetService_No,-1.1138,0.0382,0.0268
